# Predict Ahlqvist classification
Analyais of pre-COVID19 UAB data by Ahlqvist classification showed racial differences in cluster distribution. Map cluster assignments from pre-COVID19 data to post-COVID19 data.

**Objective:** Construct a machine learning model to predict cluster assignment.

# Environment setup
Import required modules.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report

from sklearn.utils import resample

Set random seed for reproducibility.

In [2]:
rng = np.random.RandomState(42)

Import cleaned data.

In [3]:
data_path = "/data/user/brianlu/projects/precision_diabetes/Clustering/exports/manuscript_Ahlqvist_analysis_2024-05-03/outpatient.csv"
data = pd.read_csv(data_path)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Columns: 300 entries, DE_ID to aa_factor
dtypes: float64(93), int64(41), object(166)
memory usage: 2.7+ MB


Set up train-test split.

In [4]:
predictors = ["gad", "a1c", "bmi", "diabetes_age", "homa_cpeptide_b", "homa_cpeptide_ir"]

# multiple classes to predict, transform into labels
le = LabelEncoder()
x = data[predictors]
le.fit(data["Ahlqvist"])
le.classes_ = np.array(["SAID", "SIDD", "SIRD", "MOD", "MARD"])
y_ahlqvist = le.transform(data["Ahlqvist"])

x_train, x_test, y_train, y_test = train_test_split(x, y_ahlqvist, test_size = 0.2, shuffle = True, random_state = rng)

# Model selection

Random forest model was chosen for simplicity. Best model parameters were selected by GridSearch with 5-fold cross validation repeated 5 times refit using weighted F1 scores.

In [5]:
cv = RepeatedKFold(n_splits = 5, n_repeats = 5, random_state = rng)
refit = "f1_weighted"

Weighted scores were used to accoung for class imbalance. Scores for each class were also inspected for each model.

In [6]:
score = ["balanced_accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_ovr_weighted"]

Function to extract scoring metrics.

In [7]:
def extract_scores(model):
    # find index of best model
    index = np.where(model.cv_results_["rank_test_f1_weighted"] == 1)[0][0]
    
    # extract scores of best model
    accuracy = model.cv_results_["mean_test_balanced_accuracy"][index]
    precision = model.cv_results_["mean_test_precision_weighted"][index]
    recall = model.cv_results_["mean_test_recall_weighted"][index]
    f1 = model.cv_results_["mean_test_f1_weighted"][index]
    roc = model.cv_results_["mean_test_roc_auc_ovr_weighted"][index]
    
    out = pd.DataFrame({"metric" : ["accuracy", "precision", "recall", "f1", "roc"], 
                        "scores" : [accuracy, precision, recall, f1, roc]})
    
    return out

In [8]:
%%time

from sklearn.ensemble import RandomForestClassifier

parameters = {"n_estimators" : [100, 250, 500],
              "max_depth" : [10, 20, 30, None],
              'min_samples_split' : range(2, 8, 2),
              "max_features" : range(1, 6)}

rf_clf = RandomForestClassifier(random_state = rng, n_jobs = -1)
rf_gs = GridSearchCV(estimator = rf_clf, 
                 param_grid = parameters,
                 cv = cv,
                 scoring = score,
                 refit = refit)

rf_gs.fit(x_train, y_train)

# save score for comparision
rf_score = extract_scores(rf_gs)

CPU times: user 14min 31s, sys: 1min 59s, total: 16min 30s
Wall time: 24min 19s


In [9]:
print(rf_gs.best_params_)
print(rf_gs.best_estimator_)

{'max_depth': 30, 'max_features': 1, 'min_samples_split': 2, 'n_estimators': 250}
RandomForestClassifier(max_depth=30, max_features=1, n_estimators=250,
                       n_jobs=-1,
                       random_state=RandomState(MT19937) at 0x2AAAAAB30240)


Cross-validation scores.

In [10]:
pd.DataFrame({"metric": rf_score["metric"],
              "random_forest" : rf_score["scores"],
              })

,metric,random_forest
0,accuracy,0.909650
1,precision,0.903096
2,recall,0.901152
3,f1,0.901171
4,roc,0.988356


Per class score after training.

In [11]:
y_train_pred = rf_gs.best_estimator_.predict(x_train)

print(classification_report(y_train, 
                      y_train_pred,
                           labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00       235
           2       1.00      1.00      1.00       133
           3       1.00      1.00      1.00       300
           4       1.00      1.00      1.00       205

    accuracy                           1.00       955
   macro avg       1.00      1.00      1.00       955
weighted avg       1.00      1.00      1.00       955



Boostrap for score confidence intervals.

In [12]:
bootstrap_n = 5000

def bootstrap_metrics(model, x, y, n, random_state = rng):
    
    # define scores to collect
    bootstrap_accuracy = list()
    bootstrap_precision = list()
    bootstrap_recall = list()
    bootstrap_f1 = list()

    # bootstrap with replacement
    for i in range(n):
        x_resampled, y_resampled = resample(x, y, random_state = random_state)
        resampled_pred = model.predict(x_resampled)
        bootstrap_accuracy.append(balanced_accuracy_score(y_resampled, resampled_pred))
        bootstrap_precision.append(precision_score(y_resampled, resampled_pred, average = "weighted"))
        bootstrap_recall.append(recall_score(y_resampled, resampled_pred, average = "weighted"))
        bootstrap_f1.append(f1_score(y_resampled, resampled_pred, average = "weighted"))

    # append model score with 95% confidence interval
    y_pred = model.predict(x)
    accuracy_col = np.insert(np.percentile(bootstrap_accuracy, [2.5, 97.5]), 0, 
                             balanced_accuracy_score(y, y_pred))
    precision_col = np.insert(np.percentile(bootstrap_precision, [2.5, 97.5]), 0,
                             precision_score(y, y_pred, average = "weighted"))
    recall_col = np.insert(np.percentile(bootstrap_recall, [2.5, 97.5]), 0,
                             recall_score(y, y_pred, average = "weighted"))
    f1_col = np.insert(np.percentile(bootstrap_f1, [2.5, 97.5]), 0, 
                       f1_score(y, y_pred, average = "weighted"))

    # return results
    bootstrap_results = pd.DataFrame({"Score" : ["Input", "2.5%", "97.5%"],
                                      "Accuracy" : accuracy_col,
                                      "Precision" : precision_col,
                                      "Recall" : recall_col,
                                      "F1" : f1_col})
    return bootstrap_results
    

In [13]:
%%time

bootstrap_metrics(rf_gs.best_estimator_, x_train, y_train, bootstrap_n)

CPU times: user 5min 6s, sys: 38.2 s, total: 5min 44s
Wall time: 3min 48s


,Score,Accuracy,Precision,Recall,F1
0,Input,1.0,1.0,1.0,1.0
1,2.5%,1.0,1.0,1.0,1.0
2,97.5%,1.0,1.0,1.0,1.0


# Model validation
Validate model with hold-out test data.

In [14]:
y_test_pred = rf_gs.best_estimator_.predict(x_test)

print(classification_report(y_test, 
                            y_test_pred,
                            labels = [0,1,2,3,4]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       0.92      0.96      0.94        50
           2       0.89      0.86      0.88        37
           3       0.91      0.93      0.92        80
           4       0.95      0.91      0.93        44

    accuracy                           0.93       239
   macro avg       0.94      0.93      0.93       239
weighted avg       0.93      0.93      0.93       239



In [15]:
pd.DataFrame({"Cluster" : ["SAID", "SIDD", "SIRD", "MOD", "MARD"],
              "precision" : precision_score(y_test, y_test_pred, average = None),
              "recall" : recall_score(y_test, y_test_pred, average = None),
              "f1" : f1_score(y_test, y_test_pred, average = None),
              "roc" : roc_auc_score(y_test, 
                                    rf_gs.best_estimator_.predict_proba(x_test), 
                                    average = None, multi_class = "ovr")})


,Cluster,precision,recall,f1,roc
0,SAID,1.000000,1.000000,1.000000,1.000000
1,SIDD,0.923077,0.960000,0.941176,0.992275
2,SIRD,0.888889,0.864865,0.876712,0.990701
3,MOD,0.913580,0.925000,0.919255,0.981289
4,MARD,0.952381,0.909091,0.930233,0.993473


In [16]:
%%time

bootstrap_metrics(rf_gs.best_estimator_, x_test, y_test, bootstrap_n)

CPU times: user 3min 45s, sys: 45.7 s, total: 4min 30s
Wall time: 3min 24s


,Score,Accuracy,Precision,Recall,F1
0,Input,0.931791,0.929012,0.928870,0.928736
1,2.5%,0.898300,0.896785,0.895397,0.894900
2,97.5%,0.961888,0.960107,0.958159,0.958356


# Remap clusters

In [17]:
post_covid_data_path = "/data/user/brianlu/projects/precision_diabetes/COVID19_clusters/exports/clustering_Ahlqvist_2024-07-19/clustering_outpatient_cleaned.csv"
post_covid = pd.read_csv(post_covid_data_path)

post_covid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Columns: 268 entries, DE_ID to Ahlqvist
dtypes: float64(104), int64(14), object(150)
memory usage: 433.5+ KB


In [18]:
# map cluster assignments based on trained model
mapped_clusters = rf_gs.best_estimator_.predict(post_covid[predictors])

# convert encoded labels back to cluster names
mapped_clusters = le.inverse_transform(mapped_clusters)

# insert back into data
post_covid["mapped_cluster"] = mapped_clusters

In [19]:
print(classification_report(post_covid["mapped_cluster"], 
                            post_covid["Ahlqvist"]))

              precision    recall  f1-score   support

        MARD       0.61      0.96      0.75        23
         MOD       0.67      0.77      0.72        56
        SAID       1.00      1.00      1.00        11
        SIDD       0.85      0.79      0.82        71
        SIRD       0.90      0.59      0.71        46

    accuracy                           0.77       207
   macro avg       0.81      0.82      0.80       207
weighted avg       0.79      0.77      0.77       207



Mapped cluster assignment have decent aggrement with de novo clustering, but not complete agreement.

# Export results

Create export directory.

In [20]:
import datetime
import os
import pickle

# append export date
date = str(datetime.date.today())

# define export directory
export_dir_path = "/data/user/brianlu/projects/precision_diabetes/COVID19_clusters/exports/remap_clusters_rf_" + date + "/"

# create export directory if folder doesn't exist
if not os.path.isdir(export_dir_path):
    os.mkdir(export_dir_path)

Export model and results.

In [21]:
# export model
model_path = export_dir_path + "rf_model.pickle"
pickle.dump(rf_gs.best_estimator_, open(model_path, 'wb')) # wb for write binary

# expor results
post_covid.to_csv(export_dir_path + "remapped_outpatient.csv", index = False)